# Reaction--diffusion equation on a Swiss-cheese surface

Semi-implicit two-species reaction--diffusion experiment on an implicit triangular surface. The notebook exports ParaView VTU snapshots for the initial condition and two later times.


In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

project = Path.cwd().resolve()
if project.name == 'notebooks':
    project = project.parent
sys.path.insert(0, str(project))

import pysurfacefun as psf

output_dir = project / 'notebook_outputs'
output_dir.mkdir(exist_ok=True)


## Geometry


In [ ]:
mesh_file = project / 'notebook_data' / 'swiss_chesse_shape.mat'
n = 10
mesh_refinement = 0
vtu_n = n


def phi(p):
    x, y, z = p[0], p[1], p[2]
    return (
        (x*x + y*y - 4.0)**2
        + (z*z - 1.0)**2
        + (y*y + z*z - 4.0)**2
        + (x*x - 1.0)**2
        + (z*z + x*x - 4.0)**2
        + (y*y - 1.0)**2
        - 15.0
    )


def dphi(p):
    x, y, z = p[0], p[1], p[2]
    return np.array([
        4.0*x*(-9.0 + 3.0*x*x + y*y + z*z),
        4.0*y*(-9.0 + x*x + 3.0*y*y + z*z),
        4.0*z*(-9.0 + x*x + y*y + 3.0*z*z),
    ])


dom = psf.LevelSetSurface(
    mesh_file,
    phi,
    dphi,
    n=n,
    nref=mesh_refinement,
    orient_outward=True,
)

residual_phi = max(
    np.max(np.abs([phi(np.array([xj, yj, zj])) for xj, yj, zj in zip(x, y, z)]))
    for x, y, z in zip(dom.x, dom.y, dom.z)
)

print(f'patches        : {dom.npatches}')
print(f'degree         : {dom.degree}')
print(f'nodes per patch: {dom.x[0].size}')
print(f'area           : {psf.tri_surfacearea(dom):.12f}')
print(f'max |phi|      : {residual_phi:.3e}')


## Parameters


In [ ]:
delta_v = 5.0e-3
delta_u = 0.516 * delta_v
alpha = 0.899
beta = -0.91
gamma = -0.899
tau1 = 0.02
tau2 = 0.2
dt = 0.1

kend = 2000
print_every = 100
snapshot_steps = (0, 1000, 2000)
snapshot_set = set(snapshot_steps)


def Nu(u, v):
    return alpha*u*(1.0 - tau1*(v**2)) + v*(1.0 - tau2*u)


def Nv(u, v):
    return beta*v*(1.0 + (alpha/beta)*tau1*u*v) + u*(gamma + tau2*v)


## Initial Condition


In [ ]:
bb = psf.boundingbox(dom)
f = psf.randnfun3(0.3, bb, seed=0)
u = psf.tri_surfacefun(lambda x, y, z: f(x, y, z), dom)
f = psf.randnfun3(0.3, bb, seed=1)
v = psf.tri_surfacefun(lambda x, y, z: f(x, y, z), dom)

print(f'|u|_inf = {u.norm_inf():.3e}')
print(f'|v|_inf = {v.norm_inf():.3e}')


## VTU Export


In [ ]:
def export_snapshot(step, field):
    filename = output_dir / f'swiss_cheese_reaction_diffusion_u_{step:04d}.vtu'
    try:
        psf.write_triangle_meshio(filename, field, point_name='u', nvis=vtu_n)
    except ImportError:
        psf.write_tri_vtu(filename, field, point_name='u', nvis=vtu_n)
    print(f'wrote {filename}')


snapshots = {0: u.copy()}
export_snapshot(0, snapshots[0])


## Build Operators


In [ ]:
t0 = time.perf_counter()
Lu = psf.tri_surfaceop(dom, {'lap': -dt*delta_u, 'b': 1.0}, 0.0)
Lu.build()
print(f'Lu build time: {time.perf_counter() - t0:.2f} s')

t0 = time.perf_counter()
Lv = psf.tri_surfaceop(dom, {'lap': -dt*delta_v, 'b': 1.0}, 0.0)
Lv.build()
print(f'Lv build time: {time.perf_counter() - t0:.2f} s')


## Time Stepping


In [ ]:
t = 0.0
for k in range(1, kend + 1):
    t0 = time.perf_counter()
    rhs_u = u + dt*Nu(u, v)
    rhs_v = v + dt*Nv(u, v)
    u = Lu.apply(rhs_u)
    v = Lv.apply(rhs_v)
    t += dt

    if k % print_every == 0:
        print(
            f'k = {k:5d}, t = {t:8.2f}, '
            f'step time = {time.perf_counter() - t0:.2f} s, '
            f'|u|_inf = {u.norm_inf():.3e}'
        )

    if k in snapshot_set:
        snapshots[k] = u.copy()
        export_snapshot(k, snapshots[k])


## Output Files

Open these files in ParaView:

- `notebook_outputs/swiss_cheese_reaction_diffusion_u_0000.vtu`
- `notebook_outputs/swiss_cheese_reaction_diffusion_u_1000.vtu`
- `notebook_outputs/swiss_cheese_reaction_diffusion_u_2000.vtu`
